# Generate data

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from mothernet.utils import generate_data

In [ ]:
data, _, y_col_name = generate_data(dataset_size = 150)
# TODO normalize dataset
data

In [ ]:
category_colors = {0: 'red', 1: 'blue'}
treatment_colors = [category_colors[t] for t in data["treatment"]]

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
plt.scatter(data['age'], data['temperature'], data['y'], c=treatment_colors, marker='o')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=data["age"], y=data["temperature"], z=data[y_col_name], mode="markers", 
                           marker={
                               "color": data["treatment"],
                               "colorbar": {"title": "treatment"}
                               }))
fig.update_layout(
    title='data',
    scene=dict(
        xaxis_title='age',
        yaxis_title='temperature',
        zaxis_title=y_col_name,
    )
)
fig.write_html("data.html")


# Define layers

In [ ]:
import torch
import torch.nn as nn

# Define the parameters
exclude_cols = ["treatment"]
embedding_dim = len(data.columns) - len(exclude_cols) - 1  # Size of each input embedding (max)
num_heads = 1  # Number of attention heads
num_layers = 6  # Number of Transformer encoder layers
dim_feedforward = 2048  # Dimension of the feedforward network model
dropout = 0.1  # Dropout rate

# Create a TransformerEncoderLayer and TransformerEncoder
encoder_layer = nn.TransformerEncoderLayer(
    d_model=embedding_dim, 
    nhead=num_heads, 
    dim_feedforward=dim_feedforward, 
    dropout=dropout
)
transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

# Dummy input tensor (sequence_length, batch_size, embedding_dim)
sequence_length = 10
batch_size = 1
x = torch.randn(sequence_length, batch_size, embedding_dim)

print("Output shape:", transformer_encoder(x).shape)  # Should be (sequence_length, batch_size, embedding_dim)

# Propagate through model

In [ ]:
data_x = data.drop(columns=[*exclude_cols, y_col_name])
data_x_tensor = torch.tensor(data_x.to_numpy().reshape(data_x.shape[0], 1, data_x.shape[1]))
output = transformer_encoder(data_x_tensor.to(torch.float32))
print(output.shape)